# GenAI Pipeline — Award Purpose Testing Notebook

LLM-based award purpose labelling of grants on alternative proteins.
Uses Claude with prompt caching via the Anthropic Python SDK. Adapted from
`grant_endproduct_testing.ipynb` — see `5_award_purpose/v1/NOTE.txt` for how the prompt
was written.


### 1. Imports and Configuration


In [2]:
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import create_model

load_dotenv()

OUTPUT_DIR = Path(".")
RAW_SUBS_DIR = Path("5_award_purpose/raw_subsets/")


### 2. Data Inspection


In [3]:
EXCEL_PATH = Path("1_deduplication/raw_data/Funding2026_inscope.xlsx")
df_raw = pd.read_excel(EXCEL_PATH)

# Almost all in-scope records have an Award purpose label (1675/1678), and this pipeline
# stage isn't split by AP pillar, so every in-scope record is eligible regardless of
# production platform — just require an Award purpose label (to sample/score against) and
# a non-empty Abstract (so the LLM has enough text to classify). Records with a label but
# no abstract are set aside for manual review instead of silently dropped.
has_awardpurpose = df_raw["Award purpose"].notna() & (df_raw["Award purpose"].str.strip() != "")
has_abstract = df_raw["Abstract"].notna() & (df_raw["Abstract"].str.strip() != "")

df = df_raw[has_awardpurpose & has_abstract].reset_index(drop=True)
needs_manual_review = df_raw[has_awardpurpose & ~has_abstract].reset_index(drop=True)

print(f"Raw shape: {df_raw.shape}")
print(f"Filtered shape (Award purpose not empty, has abstract): {df.shape}")
print(f"Set aside for manual review (label present, no abstract): {needs_manual_review.shape[0]}")
print(f"\nColumns: {list(df.columns)}")
df.head()


Raw shape: (1678, 81)
Filtered shape (Award purpose not empty, has abstract): (922, 81)
Set aside for manual review (label present, no abstract): 753

Columns: ['Title', 'Abstract', 'Original title', 'Database', 'Total amount', 'Gov contribution', 'Currency', 'Total amount (USD)', 'Gov contribution (USD)', 'Total amount (EUR)', 'Gov & NP contribution (EUR)', 'Funding decision', 'copy to external database', 'URL for announcement', 'Identification code', 'Unnamed: 15', 'Unnamed: 16', 'Notes (external)', 'Notes (internal)', 'Project lead (PI)', 'PI department', 'PI organisation', 'PI organisation type', 'PI organisation country', 'PI organisation region', 'PI organisation state', 'PI organisation zip code', 'PI organisation congressional district', 'Collaborator names', 'Collaborator institutions', 'Multiple organisation recipients', 'Date request submitted', 'Year request submitted', 'Date award announced', 'Project start date', 'Duration of award (months)', 'Project status', 'Annual exp

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
0,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,4333333.333,4333333.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,1038142.857,1038142.857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, ...",NaN,airtable,38000000,16001352,GBP,48859450.0,18697500.0,45220000.0,...,3173601.480,3173601.480,3173601.48,3173601.48,NaN,NaN,NaN,NaN,NaN,NaN


### 3. Balanced Subset Creation

Take up to `N_PER_CATEGORY` records per Award purpose category. A multi-label row (e.g.
"Research and development, Education and training") is eligible under each of its labels and is
kept only once in the final set if selected more than once.

This stage isn't split by AP pillar / production platform — every in-scope, labelled record is
eligible regardless of platform.

Also top up the sample so at least `MIN_PER_COMBO` examples of each distinct multi-label
combination are included — this specifically tests the LLM's multi-label (true/false per
category) classification. Note: `Research infrastructure` only has 4 ground-truth rows total, so
expect a "only 4 available" warning for it rather than the full `N_PER_CATEGORY`.


In [4]:
# Award purpose can hold multiple comma-separated labels per row; explode before counting.
def explode_categories(d):
    return d["Award purpose"].str.split(",").explode().str.strip()

all_categories = sorted(explode_categories(df).dropna().unique())

breakdown = explode_categories(df).value_counts().reindex(all_categories, fill_value=0)
breakdown.index.name = "award_purpose"
breakdown = breakdown.to_frame(name="count")
breakdown


,count
award_purpose,
Education and training,78
Equipment and infrastructure,11
Networking,9
Research and development,900
Research infrastructure,4


In [5]:
RANDOM_STATE = 3
N_PER_CATEGORY = 20
MIN_PER_COMBO = 2  # ensure at least this many examples of each distinct multi-label combination

def parse_categories(series):
    """Split a comma-separated 'Award purpose' string into a cleaned list of labels."""
    return series.str.split(",").apply(lambda labels: [l.strip() for l in labels])

def create_balanced_sample(df, categories, n_per_category=N_PER_CATEGORY, min_per_combo=MIN_PER_COMBO, random_state=RANDOM_STATE):
    """
    Samples up to n_per_category rows per individual Award purpose label.
    A multi-label row (e.g. "Research and development, Education and training") is eligible
    under each of its labels, and is kept only once in the combined sample if picked more than
    once. Then tops up the sample so at least min_per_combo rows of each distinct multi-label
    combination are present, to specifically test multi-label classification.
    """
    labels = parse_categories(df["Award purpose"])

    samples = []
    for cat in categories:
        mask = labels.apply(lambda xs: cat in xs)
        subset = df[mask]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        n = min(n_per_category, available)
        if n < n_per_category:
            print(f"  Warning: '{cat}' — requested {n_per_category} but only {available} available, taking all.")
        samples.append(subset.sample(n=n, random_state=random_state))

    combined = pd.concat(samples) if samples else df.iloc[0:0]
    combined = combined[~combined.index.duplicated(keep="first")]

    multi_mask = labels.apply(lambda xs: len(xs) > 1)
    multi_labels = labels[multi_mask]
    combos = multi_labels.apply(lambda xs: ", ".join(sorted(xs)))

    for combo in combos.unique():
        combo_idx = combos[combos == combo].index
        group = df.loc[combo_idx]
        already = group.index.isin(combined.index).sum()
        need = min_per_combo - already
        if need <= 0:
            continue
        remaining_pool = group[~group.index.isin(combined.index)]
        take = min(need, len(remaining_pool))
        if take < need:
            print(f"  Warning: combo '{combo}' — only {already + len(remaining_pool)} rows available, wanted {min_per_combo}.")
        if take > 0:
            combined = pd.concat([combined, remaining_pool.sample(n=take, random_state=random_state)])

    combined = combined[~combined.index.duplicated(keep="first")]
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)


In [6]:
awardpurpose_categories = breakdown.index.tolist()
test_data = create_balanced_sample(df, awardpurpose_categories)
print(f"test_data: {test_data.shape}")
# test_data[["Title", "Abstract", "Award purpose"]]


test_data: (62, 81)


### 4. Save Subset to Excel
Allows manual check of files selected. Consider whether those in the test set are borderline cases or clear cut.


In [7]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=RAW_SUBS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

save_subset(test_data, f"awardpurpose_test_data_rand{RANDOM_STATE}.xlsx")


Saved 62 records to 5_award_purpose\raw_subsets\awardpurpose_test_data_rand3.xlsx


### 5. Load Prompt and Select Dataset


In [8]:
PROMPT_VERSION = "v1"  # ← CHANGE THIS to switch prompt version
PROMPT_PATH = f"5_award_purpose/{PROMPT_VERSION}/prompt_awardpurpose_grants_{PROMPT_VERSION}.md"

DATASET = test_data

# ONLY USED AS REQUIRED FOR RE-RUN SPECIFIC RECORDS
#ids_to_test = [0, 1]
#DATASET = DATASET[DATASET["id"].isin(ids_to_test)]
#DATASET = incorrect_awardpurpose_data
#DATASET = manual_test_data


In [9]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative protein grant funding.

Your task is to classify a grant on alternative proteins against a set of award purpose categories based on both its title and abstract.

Before assessing categories, identify every purpose this award serves. A grant very often serves more than one purpose at once — for example, a national centre grant can simultaneously fund research and development, train researchers, build a shared research facility, and coordinate a network of partners. In that case, flag every category that genuinely applies.

IMPORTANT: Base your classification ONLY on what the title and abstract explicitly state about what the award funds. Research and development is the default purpose of almost every alternative protein grant — flag it True for the overwhelming majority of grants, and only withhold it when the grant exclusively funds one of the other four purposes with no described research or development activity of its own.

Key decision rules:
- Equip

### 6. API Call with Prompt Caching


In [10]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records. A dedicated
# subfolder keeps these runs from colliding with other stages' checkpoints.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints/awardpurpose")
RESUME_INCOMPLETE = True


In [11]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

AWARDPURPOSE_CATS = [
    "Research and development",
    "Education and training",
    "Networking",
    "Equipment and infrastructure",
    "Research infrastructure",
]

import re

def make_schema(cats, include_reasoning):
    """
    Builds a schema with one boolean field per category (true/false, multi-label)
    instead of a single primary/secondary pick, since a grant can serve more than
    one award purpose at once. field_map translates the sanitised Python-safe
    field names (e.g. "Research_and_development") back to the original category
    label (e.g. "Research and development").
    """
    def field_name(cat):
        return re.sub(r"\W+", "_", cat).strip("_")

    field_map = {field_name(cat): cat for cat in cats}
    fields = {fname: (bool, ...) for fname in field_map}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    Model = create_model("ClassificationSchema", **fields)
    return Model, field_map

ClassificationSchema, CATEGORY_FIELD_MAP = make_schema(AWARDPURPOSE_CATS, INCLUDE_REASONING)
print(f"Schema built: {AWARDPURPOSE_CATS}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output


Schema built: ['Research and development', 'Education and training', 'Networking', 'Equipment and infrastructure', 'Research infrastructure']


### 7. Error Handling with Retry


In [12]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


In [13]:
def classify_with_error_handling(row, system_prompt):
    record_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {record_id}: model returned no structured output")
                return {"id": record_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["id"] = record_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {record_id}: {last_error}")
    return {"id": record_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers


In [14]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Test Data


In [15]:
# Standardise DATASET column names once, separately from the LLM-calling loop below,
# so this (and the comparison cell) can be re-run for free without re-hitting the API
# — e.g. after a kernel restart, or when re-analysing results already in memory / a
# checkpoint file. Grants data has no reliable unique id column, so use row position.
DATASET = DATASET.reset_index(drop=True).rename(columns={"Title": "title", "Abstract": "abstract"})
DATASET["id"] = DATASET.index


In [16]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)



Run 1 / 1
  [1/62] 0
  [2/62] 1
  [3/62] 2
  [4/62] 3
  [5/62] 4
  [6/62] 5
  [7/62] 6
  [8/62] 7
  [9/62] 8
  [10/62] 9
  [11/62] 10
  [12/62] 11
  [13/62] 12
  [14/62] 13
  [15/62] 14
  [16/62] 15
  [17/62] 16
  [18/62] 17
  [19/62] 18
  [20/62] 19
  [21/62] 20
  [22/62] 21
  [23/62] 22
  [24/62] 23
  [25/62] 24
  [26/62] 25
  [27/62] 26
  [28/62] 27
  [29/62] 28
  [30/62] 29
  [31/62] 30
  [32/62] 31
  [33/62] 32
  [34/62] 33
  [35/62] 34
  [36/62] 35
  [37/62] 36
  [38/62] 37
  [39/62] 38
  [40/62] 39
  [41/62] 40
  [42/62] 41
  [43/62] 42
  [44/62] 43
  [45/62] 44
  [46/62] 45
  [47/62] 46
  [48/62] 47
  [49/62] 48
  [50/62] 49
  [51/62] 50
  [52/62] 51
  [53/62] 52
  [54/62] 53
  [55/62] 54
  [56/62] 55
  [57/62] 56
  [58/62] 57
  [59/62] 58
  [60/62] 59
  [61/62] 60
  [62/62] 61


In [17]:
results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Completed: 62 records across 1 run(s)
Successful: 62
Errors: 0


,Research_and_development_LLM,Education_and_training_LLM,Networking_LLM,Equipment_and_infrastructure_LLM,Research_infrastructure_LLM,reasoning_LLM,id,status,run
0,True,False,False,False,False,This grant funds scientific research and devel...,0,ok,1
1,True,False,False,False,False,This grant funds scientific research and devel...,1,ok,1
2,True,False,False,False,False,This grant funds scientific research and devel...,2,ok,1
3,True,True,False,False,False,This is a Doctoral Training Partnership grant ...,3,ok,1
4,False,False,True,False,True,The grant explicitly funds the creation of an ...,4,ok,1
...,...,...,...,...,...,...,...,...,...
57,True,True,False,False,False,This is a studentship (PhD training) grant tha...,57,ok,1
58,True,True,False,False,False,This grant funds scientific research comparing...,58,ok,1
59,True,False,False,False,False,The grant funds the development and launch of ...,59,ok,1
60,False,False,False,True,False,The grant exclusively funds scaling up operati...,60,ok,1


In [18]:
def build_predictions(row):
    true_cats = [orig for field, orig in CATEGORY_FIELD_MAP.items() if row.get(f"{field}_LLM") == True]
    # Joined with "; " for consistency with the sibling notebooks' convention (some of their
    # category names contain commas, which would make a comma-joined string ambiguous to
    # split back apart) — none of these 5 category names contain a comma, but keeping the
    # same separator everywhere avoids a one-off inconsistency in the pipeline.
    return pd.Series({
        "raw_predicted_categories": "; ".join(true_cats),
        "predicted_award_purpose": "; ".join(true_cats),
    })

results_df[["raw_predicted_categories", "predicted_award_purpose"]] = results_df.apply(build_predictions, axis=1)

result_cols = ["id", "run", "raw_predicted_categories", "predicted_award_purpose", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "Award purpose"]].merge(
    results_df[result_cols], on="id", how="left"
)
comparison = comparison.rename(columns={"Award purpose": "award_purpose"})

# Ground truth is comma-separated (its raw data format); our own predictions are
# semicolon-separated (see build_predictions above). Compare both as sets so label order
# doesn't affect the match.
def to_label_set(s, sep=","):
    if not isinstance(s, str) or not s.strip():
        return set()
    return {l.strip() for l in s.split(sep)}

comparison["award_purpose_set"] = comparison["award_purpose"].apply(lambda s: to_label_set(s, sep=","))
comparison["predicted_award_purpose_set"] = comparison["predicted_award_purpose"].apply(lambda s: to_label_set(s, sep=";"))
comparison["exact_match"] = comparison["award_purpose_set"] == comparison["predicted_award_purpose_set"]

# Partial-credit metrics per row: even when the full label set doesn't match exactly,
# how much overlap is there between predicted and true labels?
def label_prf(row):
    truth, pred = row["award_purpose_set"], row["predicted_award_purpose_set"]
    if not truth and not pred:
        return pd.Series({"row_precision": 1.0, "row_recall": 1.0, "row_jaccard": 1.0})
    tp = len(truth & pred)
    fp = len(pred - truth)
    fn = len(truth - pred)
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    jaccard = tp / len(truth | pred) if (truth | pred) else float("nan")
    return pd.Series({"row_precision": precision, "row_recall": recall, "row_jaccard": jaccard})

comparison[["row_precision", "row_recall", "row_jaccard"]] = comparison.apply(label_prf, axis=1)

n = len(comparison)
print("== Award purpose ==")
print(f"Exact match accuracy:  {comparison['exact_match'].mean():.0%}  (n={n})")
print(f"Mean row precision:    {comparison['row_precision'].mean():.0%}")
print(f"Mean row recall:       {comparison['row_recall'].mean():.0%}")
print(f"Mean row Jaccard:      {comparison['row_jaccard'].mean():.0%}")

# Per-category precision/recall across the multi-label predictions
def label_series(s, sep=","):
    return s.str.split(sep).explode().str.strip().dropna()

all_cats = sorted(set(label_series(comparison["award_purpose"], sep=",")) | set(label_series(comparison["predicted_award_purpose"], sep=";")))
cat_rows = []
for cat in all_cats:
    truth_has = comparison["award_purpose_set"].apply(lambda s: cat in s)
    pred_has  = comparison["predicted_award_purpose_set"].apply(lambda s: cat in s)
    tp = int((truth_has & pred_has).sum())
    fn = int((truth_has & ~pred_has).sum())
    fp = int((~truth_has & pred_has).sum())
    n_true = int(truth_has.sum())
    recall = tp / n_true if n_true else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    cat_rows.append({
        "award_purpose": cat, "n_true": n_true, "tp": tp, "fp": fp, "fn": fn,
        "recall": recall, "precision": precision,
    })
cat_stats = pd.DataFrame(cat_rows).set_index("award_purpose")
cat_stats["recall"] = cat_stats["recall"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
cat_stats["precision"] = cat_stats["precision"].map(lambda x: f"{x:.0%}" if pd.notna(x) else "-")
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "award_purpose", "raw_predicted_categories", "predicted_award_purpose"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["exact_match", "row_precision", "row_recall", "row_jaccard"]
comparison[display_cols]


== Award purpose ==
Exact match accuracy:  69%  (n=62)
Mean row precision:    92%
Mean row recall:       91%
Mean row Jaccard:      84%


,n_true,tp,fp,fn,recall,precision
award_purpose,,,,,,
Education and training,23,13,1,10,57%,93%
Equipment and infrastructure,11,11,0,0,100%,100%
Networking,9,9,4,0,100%,69%
Research and development,46,45,6,1,98%,88%
Research infrastructure,4,3,0,1,75%,100%


,id,title,abstract,award_purpose,raw_predicted_categories,predicted_award_purpose,reasoning_LLM,exact_match,row_precision,row_recall,row_jaccard
0,0,Reimagine protein - Processing of alternative ...,The state of our planet and environment is cur...,Research and development,Research and development,Research and development,This grant funds scientific research and devel...,True,1.0,1.0,1.0
1,1,A New Approach to the Production of Cultured M...,Future food production needs to meet a growing...,"Research and development, Education and training",Research and development,Research and development,This grant funds scientific research and devel...,False,1.0,0.5,0.5
2,2,APPLICATION OF THERMOEXTRUSION TO OBTAIN A TEX...,"Currently, one of the most impactful trends in...",Research and development,Research and development,Research and development,This grant funds scientific research and devel...,True,1.0,1.0,1.0
3,3,Utilising food waste as a feedstock for cultiv...,Doctoral Training Partnerships: a range of pos...,"Research and development, Education and training",Research and development; Education and training,Research and development; Education and training,This is a Doctoral Training Partnership grant ...,True,1.0,1.0,1.0
4,4,Federating Irish research infrastructures to a...,This project will create an all-island Irish n...,"Research infrastructure, Networking",Networking; Research infrastructure,Networking; Research infrastructure,The grant explicitly funds the creation of an ...,True,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
57,57,Biorefining Protein from UK Grasslands - Can W...,British grassland farming covers 72% (12.6 mil...,"Research and development, Education and training",Research and development; Education and training,Research and development; Education and training,This is a studentship (PhD training) grant tha...,True,1.0,1.0,1.0
58,58,Comparing the Physical Properties of Plant Lip...,Comparing the Physical Properties of Plant Lip...,"Research and development, Education and training",Research and development; Education and training,Research and development; Education and training,This grant funds scientific research comparing...,True,1.0,1.0,1.0
59,59,BettaF!sh,Through the EIT Fast Track to Market initiativ...,Research and development,Research and development,Research and development,The grant funds the development and launch of ...,True,1.0,1.0,1.0
60,60,Scaling up to a new mycoprotein factory,The funding will be used to scale up its opera...,Equipment and infrastructure,Equipment and infrastructure,Equipment and infrastructure,The grant exclusively funds scaling up operati...,True,1.0,1.0,1.0


### 10. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign award purpose labels, save the
comparison data, then manually review what went wrong and adjust the prompt. None of this will
make it into the final workflow.

Order of working:
1. Create a new version folder in `5_award_purpose` (e.g. `v2`).
2. Copy in the previous prompt. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 10 output directory (this step) and Step 5 prompt selection and input data.
4. Run the script from steps 5-10.
5. Manually review the results — both metrics and individual rows.
6. Write a text document (`NOTE.txt`) about the results and what changes you want to make to the prompt. Repeat from step 1.


In [19]:
save_dir = Path(f"5_award_purpose/{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "award_purpose_exact_match_accuracy", "value": f"{comparison['exact_match'].mean():.0%}", "n": n},
    {"metric": "award_purpose_mean_precision", "value": f"{comparison['row_precision'].mean():.0%}", "n": n},
    {"metric": "award_purpose_mean_recall", "value": f"{comparison['row_recall'].mean():.0%}", "n": n},
    {"metric": "award_purpose_mean_jaccard", "value": f"{comparison['row_jaccard'].mean():.0%}", "n": n},
])

out_path = save_dir / f"awardpurpose_{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")


Saved to 5_award_purpose\v1\awardpurpose_v1_claude-sonnet-4-6_results.xlsx


In [ ]:
# Records where the LLM's predicted Award purpose set did not exactly match ground truth —
# for re-run with a modified prompt
incorrect_ids = comparison.loc[~comparison["exact_match"], "id"]
incorrect_awardpurpose_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_awardpurpose_data


In [ ]:
# Manually select specific rows by id for quick re-testing (paste ids from the
# comparison/results tables above). Note: the Section 9 prep cell always resets
# "id" to a fresh 0..n-1 range when it runs, so once this subset goes through the
# script again its ids won't match the ones you selected here — use "title" or
# "award_purpose" to cross-reference back to the original run if needed.
manual_ids = [0, 1]  # <- CHANGE THIS to the ids you want to re-test
manual_test_data = DATASET[DATASET["id"].isin(manual_ids)].reset_index(drop=True)
print(f"Selected {len(manual_test_data)} of {len(manual_ids)} requested ids")
manual_test_data
